In [2]:




from pypdf import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
import google.generativeai as genai
import redis



GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"

genai.configure(api_key=GEMINI_API_KEY)

llm = genai.GenerativeModel("gemini-1.5-flash")



PDF_FILE = "resume.pdf"     # change filename

reader = PdfReader(PDF_FILE)

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text + "\n"

print("PDF Loaded Successfully")
print("Total Characters:", len(text))



splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_text(text)

print("Total Chunks:", len(chunks))



embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(chunks)

print("Embeddings Generated")
print("Embedding Shape:", embeddings.shape)



client = chromadb.Client()

try:
    client.delete_collection("pdf_rag")
except:
    pass

collection = client.create_collection(
    name="pdf_rag"
)

for i, chunk in enumerate(chunks):

    collection.add(
        ids=[str(i)],
        documents=[chunk]
    )

print("Documents Stored In ChromaDB")



cache = redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)

print("Redis Connected")



def ask_pdf(question):

    cached_answer = cache.get(question)

    if cached_answer:

        print("\n========================")
        print("ANSWER FROM CACHE")
        print("========================\n")
        print(cached_answer)
        return

    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    context = "\n\n".join(
        results["documents"][0]
    )

    prompt = f"""
You are a PDF Knowledge Assistant.

Answer ONLY from the supplied context.

If the answer is unavailable,
reply with:
'Information not found in document.'

CONTEXT:
{context}

QUESTION:
{question}

Give a short and precise answer.
"""

    response = llm.generate_content(prompt)

    answer = response.text

    cache.set(question, answer)

    print("\n========================")
    print("ANSWER FROM LLM")
    print("========================\n")
    print(answer)



ask_pdf("What is the candidate's education?")

ask_pdf("What programming languages does the candidate know?")

ask_pdf("What projects are mentioned in the resume?")

ask_pdf("What AI skills are available?")


while True:

    question = input("\nAsk Question (type exit to stop): ")

    if question.lower() == "exit":
        break

    ask_pdf(question)

PDF Loaded Successfully
Characters: 4544
Total Chunks: 11


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings Created
Shape: (11, 384)
Stored in ChromaDB

QUESTION:
What skills does the candidate have?

RETRIEVED TEXT:
■  Performed end-to-end AI/ML intern workflow: queried and cleaned structured datasets with SQL and Pandas, engineered features, trained and
 evaluated a KNN classifier in Python (confusion matrix, F1-score) — responsibilities fully aligned with AI intern role scope.
KEY PROJECTS
SkillRoad — AI-Powered Career Guidance Platform  React.js, Vite, Tailwind CSS, Node.js, MongoDB, Claude API

Question: What is the candidate's education?

Answer:
■  Gemini for Data Scientists and Analysts — Google Cloud Skills Boost  Earned May 12, 2026
 
■  Google Data Analytics Professional Certificate — Google
 
■  Claude API — AI-Assisted Development (Hands-on, Project-based) — Anthropic
 
EDUCATION
B.Tech, Computer Science Engineering (AI & Machine Intelligence)
 Expected 2027
Kalasalingam Academy of Research and Education (KARE) | CGPA: 6.28/10 | Reg. No. 99230040918
Intermediate (MPC)

In [4]:


while True:

    question = input("\nAsk your question (type 'exit' to stop): ")

    if question.lower() == "exit":
        print("Chatbot Closed")
        break

    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    context = "\n".join(results["documents"][0])

    prompt = f"""
    Answer the question using only the context below.

    Context:
    {context}

    Question:
    {question}

    Give a short and accurate answer.
    """

    response = llm.generate_content(prompt)

    print("\nAnswer:")
    print(response.text)


Ask your question (type 'exit' to stop):  what skills


NameError: name 'llm' is not defined

In [5]:
import google.generativeai as genai

genai.configure(api_key="YOUR_GEMINI_API_KEY")

llm = genai.GenerativeModel("gemini-1.5-flash")

print("LLM Loaded Successfully")

C:\Users\shann\Documents\New folder\python22\envs\langchain_env\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.10) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


LLM Loaded Successfully


C:\Users\shann\AppData\Local\Temp\ipykernel_25032\109929372.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [7]:
!pip install -U google-generativeai

In [8]:
import google.generativeai as genai

genai.configure(api_key="YOUR_GEMINI_API_KEY")

for m in genai.list_models():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

In [12]:
import google.generativeai as genai

genai.configure(api_key="YOUR_GEMINI_API_KEY")

llm = genai.GenerativeModel("gemini-flash-latest")

response = llm.generate_content("What is Artificial Intelligence?")

print(response.text)

**Artificial Intelligence (AI)** refers to the simulation of human intelligence in machines or computer systems. In simple terms, AI is about creating computers or software that can **think, learn, reason, and solve problems** in ways that typically require human brainpower.

Unlike traditional software, which can only follow strict, pre-programmed rules (e.g., "if button A is pressed, do action B"), AI systems can analyze vast amounts of data, recognize patterns, adapt to new inputs, and make decisions independently.

---

### Key Components of AI (How it Works)

AI relies on a few core technologies to function:

1. **Data:** The fuel for AI. Machines need massive amounts of information to learn.
2. **Algorithms:** The mathematical rules and instructions that tell the computer how to learn and solve problems.
3. **Machine Learning (ML):** A branch of AI where computers learn from data without being explicitly programmed for every single scenario. For example, instead of writing rules 

In [16]:
def ask_pdf(question):

    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    context = "\n".join(results["documents"][0])

    prompt = f"""
    You are a PDF assistant.

    Answer ONLY using the context below.

    Context:
    {context}

    Question:
    {question}

    If the answer is not in the context, say:
    'Information not found in document.'

    Give a concise answer.
    """

    response = llm.generate_content(prompt)

    print("\nAnswer:\n")
    print(response.text)

In [17]:
while True:

    question = input("\nAsk Question (type exit to stop): ")

    if question.lower() == "exit":
        print("Chatbot Closed")
        break

    ask_pdf(question)


Ask Question (type exit to stop):  hii



Answer:

Information not found in document.



Ask Question (type exit to stop):  skills



Answer:

Based on the provided context, the technical skills listed are:

* **React & Frontend:** React.js, JavaScript (ES6+), HTML5, CSS3, Tailwind CSS, Vite, Component Architecture, Responsive UI Design, Cross-browser Compatibility, REST API Integration
* **AI-Assisted Development:** Claude API (Anthropic), Gemini AI (Google Cloud), LLM Integration, Prompt Engineering, AI Workflow Automation, BigQuery ML, GCP
* **Databases & SQL:** MySQL, SQL (DDL/DML/Joins/Aggregations/Indexing), MongoDB (CRUD, Aggregation Pipeline), BigQuery SQL, Schema Design
* **Other Skills Mentioned:** Python, Pandas, Node.js, ML (KNN classification, feature engineering).



Ask Question (type exit to stop):  education


...


Answer:

Performed an end-to-end AI/ML intern workflow: queried and cleaned structured datasets with SQL and Pandas, engineered features, and trained and evaluated a KNN classifier in Python (using confusion matrix and F1-score).



Ask Question (type exit to stop):  CGPA



Answer:

6.28/10



Ask Question (type exit to stop):  EXIT


Chatbot Closed


In [20]:
cache = {}

def ask_pdf_cached(question):

    if question in cache:
        print("\nAnswer From Cache:\n")
        print(cache[question])
        return

    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    context = "\n".join(results["documents"][0])

    prompt = f"""
    Context:
    {context}

    Question:
    {question}
    """

    response = llm.generate_content(prompt)

    answer = response.text

    cache[question] = answer

    print("\nAnswer:\n")
    print(answer)